# Deep RL — DQN

**Reinforcement Learning Course** — Notebook 03c

| Algorithm | Environment | Key idea |
|-----------|-------------|----------|
| DQN | CartPole-v1 | Q-learning + neural approximation |

In [ ]:
%pip install -q "gymnasium[classic-control]" torch matplotlib plotly

In [2]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import gymnasium as gym
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

ModuleNotFoundError: No module named 'gymnasium'

## DQN

Parametrise $Q_w(s, a)$ with a neural network. TD target uses a **frozen copy** $Q_{w^-}$:

$$y = R_{t+1} + \gamma \max_{a'} Q_{w^-}(S_{t+1},\, a')$$

Gradient step on a random mini-batch sampled from replay buffer $\mathcal{D}$:

$$\mathcal{L}(w) = \underset{(S,A,R,S') \sim \mathcal{D}}{\mathbb{E}}\!\left[\bigl(y - Q_w(S, A)\bigr)^2\right]$$

Target weights sync: $w^- \leftarrow w$ every $C$ episodes.

In [ ]:
class QNet(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),      nn.ReLU(),
            nn.Linear(64, act_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int = 10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a: int, r: float, s_next, done: bool) -> None:
        self.buf.append((s, a, r, s_next, float(done)))

    def sample(self, batch_size: int) -> tuple:
        s, a, r, s_next, done = zip(*random.sample(self.buf, batch_size))
        return (
            torch.tensor(np.array(s),      dtype=torch.float32),
            torch.tensor(a,                dtype=torch.long),
            torch.tensor(r,                dtype=torch.float32),
            torch.tensor(np.array(s_next), dtype=torch.float32),
            torch.tensor(done,             dtype=torch.float32),
        )

    def __len__(self) -> int:
        return len(self.buf)

In [ ]:
GAMMA        = 0.99
BATCH_SIZE   = 64
LR           = 1e-3
MIN_BUFFER   = 1_000
TARGET_SYNC  = 10      # sync Q_w⁻ ← Q_w every C episodes
EPS_START    = 1.0
EPS_END      = 0.01
EPS_DECAY    = 300
N_EPISODES   = 600

env = gym.make('CartPole-v1')
obs_dim, act_dim = env.observation_space.shape[0], env.action_space.n

q_net    = QNet(obs_dim, act_dim)
q_target = QNet(obs_dim, act_dim)
q_target.load_state_dict(q_net.state_dict())
q_target.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
buffer    = ReplayBuffer()

ep_returns:  list[float] = []
eps_history: list[float] = []

for ep in range(N_EPISODES):
    eps = EPS_END + (EPS_START - EPS_END) * np.exp(-ep / EPS_DECAY)
    eps_history.append(eps)

    obs, _ = env.reset()
    total_r = 0.0

    while True:
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net(torch.tensor(obs, dtype=torch.float32)).argmax().item()

        obs_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer.push(obs, a, r, obs_next, done)
        obs = obs_next
        total_r += r

        if len(buffer) >= MIN_BUFFER:
            s_b, a_b, r_b, s_next_b, done_b = buffer.sample(BATCH_SIZE)
            with torch.no_grad():
                y = r_b + GAMMA * q_target(s_next_b).max(1).values * (1.0 - done_b)
            q_pred = q_net(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
            loss = nn.functional.mse_loss(q_pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    if ep % TARGET_SYNC == 0:
        q_target.load_state_dict(q_net.state_dict())

    ep_returns.append(total_r)
    if ep % 100 == 0:
        print(f'ep {ep:4d}  return={total_r:.0f}  eps={eps:.3f}')

env.close()
print('Done.')

In [ ]:
def smooth(x: list[float], w: int = 20) -> np.ndarray:
    return np.convolve(x, np.ones(w) / w, mode='valid')

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(
    y=ep_returns, mode='lines',
    line=dict(color='rgba(0,212,255,0.20)', width=1),
    name='return',
), secondary_y=False)
fig.add_trace(go.Scatter(
    y=smooth(ep_returns), mode='lines',
    line=dict(color='rgba(0,212,255,1)', width=2),
    name='smoothed',
), secondary_y=False)
fig.add_trace(go.Scatter(
    y=eps_history, mode='lines',
    line=dict(color='rgba(245,158,11,0.6)', width=1.5, dash='dot'),
    name='ε',
), secondary_y=True)

fig.update_layout(
    title='DQN — CartPole-v1',
    xaxis_title='episode',
    template='plotly_dark', height=350,
)
fig.update_yaxes(title_text='return',   secondary_y=False)
fig.update_yaxes(title_text='epsilon',  secondary_y=True, range=[0, 1.05])
fig.show()

In [ ]:
# Run one greedy episode and capture frames
env_render = gym.make('CartPole-v1', render_mode='rgb_array')
obs, _ = env_render.reset(seed=0)
frames = [env_render.render()]

q_net.eval()
with torch.no_grad():
    done = False
    while not done:
        a = q_net(torch.tensor(obs, dtype=torch.float32)).argmax().item()
        obs, _, terminated, truncated, _ = env_render.step(a)
        frames.append(env_render.render())
        done = terminated or truncated

env_render.close()

fig_r, ax = plt.subplots(figsize=(5, 3))
ax.axis('off')
im = ax.imshow(frames[0])

def update(i):
    im.set_data(frames[i])
    return (im,)

ani = animation.FuncAnimation(fig_r, update, frames=len(frames), interval=30, blit=True)
plt.close(fig_r)
HTML(ani.to_jshtml())